# 07 GenAI Inventory RAG Retrieval

This notebook retrieves relevant inventory context from the `inventory_rag_documents` table.

The retrieval logic combines simple keyword matching with business priority scoring.

This is the first version of the inventory RAG assistant retrieval layer.

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail_capstone")

spark.sql("SELECT current_catalog(), current_schema()").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace| retail_capstone|
+-----------------+----------------+



In [0]:
from pyspark.sql.functions import col, lower, lit, when, round

In [0]:
rag_df = spark.table("workspace.retail_capstone.inventory_rag_documents")

display(
    rag_df.select(
        "document_id",
        "stock_code",
        "product_name",
        "stockout_risk_level",
        "reorder_flag",
        "business_priority_score",
        "rag_document_text"
    )
)

document_id,stock_code,product_name,stockout_risk_level,reorder_flag,business_priority_score,rag_document_text
84029G_WH002,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,High Risk,Reorder Needed,0.909,"Product 84029G, KNITTED UNION FLAG HOT WATER BOTTLE, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 25 units. Current stock is 35 units. Average daily sales is 17.91 units. Days of inventory remaining is 1.4. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 122.5."
85123A_WH001,85123A,WHITE HANGING HEART T-LIGHT HOLDER,High Risk,No Reorder Needed,0.606,"Product 85123A, WHITE HANGING HEART T-LIGHT HOLDER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 210 units. Current stock is 250 units. Average daily sales is 120.15 units. Days of inventory remaining is 1.75. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is High Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 312.5."
84029E_WH002,84029E,RED WOOLLY HOTTIE WHITE HEART,High Risk,Reorder Needed,0.909,"Product 84029E, RED WOOLLY HOTTIE WHITE HEART, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 48 units. Current stock is 60 units. Average daily sales is 38.24 units. Days of inventory remaining is 1.26. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 195.0."
84879_WH001,84879,ASSORTED COLOUR BIRD ORNAMENT,High Risk,No Reorder Needed,0.606,"Product 84879, ASSORTED COLOUR BIRD ORNAMENT, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 100 units. Current stock is 110 units. Average daily sales is 117.5 units. Days of inventory remaining is 0.85. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is High Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 165.0."
71053_WH001,71053,WHITE METAL LANTERN,Medium Risk,Reorder Needed,0.756,"Product 71053, WHITE METAL LANTERN, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 75 units. Current stock is 90 units. Average daily sales is 10.41 units. Days of inventory remaining is 7.2. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is Medium Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 200 units. Inventory value is 189.0."
21730_WH001,21730,GLASS STAR FROSTED T-LIGHT HOLDER,Low Risk,No Reorder Needed,0.251,"Product 21730, GLASS STAR FROSTED T-LIGHT HOLDER, belongs to category Home Decor and brand GlassWorks. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 110 units. Current stock is 140 units. Average daily sales is 6.29 units. Days of inventory remaining is 17.49. Supplier is GlassWorks Studio with lead time of 9 days and reliability score of 0.89. Stockout risk level is Low Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 385.0."
22752_WH003,22752,SET 7 BABUSHKA NESTING BOXES,High Risk,Reorder Needed,0.914,"Product 22752, SET 7 BABUSHKA NESTING BOXES, belongs to category Gifts and brand GiftCraft. It is stored in warehouse WH003 named Cologne Regional Wareho

In [0]:
def retrieve_inventory_context(query: str, top_k: int = 5):
    query_lower = query.lower()

    result_df = rag_df.withColumn(
        "query",
        lit(query_lower)
    ).withColumn(
        "retrieval_score",
        when(lower(col("rag_document_text")).contains(query_lower), 1.0)
        .when(lower(col("stockout_risk_level")).contains(query_lower), 0.9)
        .when(lower(col("reorder_flag")).contains(query_lower), 0.9)
        .when(lower(col("product_name")).contains(query_lower), 0.8)
        .when(lower(col("category")).contains(query_lower), 0.7)
        .when(lower(col("supplier_name")).contains(query_lower), 0.7)
        .otherwise(0.1)
    ).withColumn(
        "final_score",
        round(
            col("retrieval_score") * (lit(0.6) + lit(0.4) * col("business_priority_score")),
            4
        )
    ).orderBy(
        col("final_score").desc(),
        col("business_priority_score").desc()
    )

    return result_df.select(
        "document_id",
        "stock_code",
        "product_name",
        "warehouse_id",
        "stockout_risk_level",
        "reorder_flag",
        "business_priority_score",
        "retrieval_score",
        "final_score",
        "rag_document_text"
    ).limit(top_k)

In [0]:
display(
    retrieve_inventory_context("Reorder Needed", top_k=5)
)

document_id,stock_code,product_name,warehouse_id,stockout_risk_level,reorder_flag,business_priority_score,retrieval_score,final_score,rag_document_text
22752_WH003,22752,SET 7 BABUSHKA NESTING BOXES,WH003,High Risk,Reorder Needed,0.914,1.0,0.9656,"Product 22752, SET 7 BABUSHKA NESTING BOXES, belongs to category Gifts and brand GiftCraft. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 20 units. Current stock is 25 units. Average daily sales is 9.2 units. Days of inventory remaining is 2.17. Supplier is GiftCraft Wholesale with lead time of 12 days and reliability score of 0.86. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 100 units. Inventory value is 100.0."
84406B_WH002,84406B,CREAM CUPID HEARTS COAT HANGER,WH002,High Risk,Reorder Needed,0.912,1.0,0.9648,"Product 84406B, CREAM CUPID HEARTS COAT HANGER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 40 units. Current stock is 45 units. Average daily sales is 12.52 units. Days of inventory remaining is 3.19. Supplier is DecorCraft Europe with lead time of 10 days and reliability score of 0.88. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 150 units. Inventory value is 83.25."
84029E_WH002,84029E,RED WOOLLY HOTTIE WHITE HEART,WH002,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 84029E, RED WOOLLY HOTTIE WHITE HEART, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 48 units. Current stock is 60 units. Average daily sales is 38.24 units. Days of inventory remaining is 1.26. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 195.0."
84029G_WH002,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,WH002,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 84029G, KNITTED UNION FLAG HOT WATER BOTTLE, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 25 units. Current stock is 35 units. Average daily sales is 17.91 units. Days of inventory remaining is 1.4. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 122.5."
22633_WH003,22633,HAND WARMER UNION JACK,WH003,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 22633, HAND WARMER UNION JACK, belongs to category Accessories and brand WarmHome. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 60 units. Current stock is 80 units. Average daily sales is 45.52 units. Days of inventory remaining is 1.32. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 300 units. Inventory value is 88.0."


In [0]:
display(
    retrieve_inventory_context("High Risk", top_k=5)
)

document_id,stock_code,product_name,warehouse_id,stockout_risk_level,reorder_flag,business_priority_score,retrieval_score,final_score,rag_document_text
22752_WH003,22752,SET 7 BABUSHKA NESTING BOXES,WH003,High Risk,Reorder Needed,0.914,1.0,0.9656,"Product 22752, SET 7 BABUSHKA NESTING BOXES, belongs to category Gifts and brand GiftCraft. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 20 units. Current stock is 25 units. Average daily sales is 9.2 units. Days of inventory remaining is 2.17. Supplier is GiftCraft Wholesale with lead time of 12 days and reliability score of 0.86. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 100 units. Inventory value is 100.0."
84406B_WH002,84406B,CREAM CUPID HEARTS COAT HANGER,WH002,High Risk,Reorder Needed,0.912,1.0,0.9648,"Product 84406B, CREAM CUPID HEARTS COAT HANGER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 40 units. Current stock is 45 units. Average daily sales is 12.52 units. Days of inventory remaining is 3.19. Supplier is DecorCraft Europe with lead time of 10 days and reliability score of 0.88. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 150 units. Inventory value is 83.25."
84029E_WH002,84029E,RED WOOLLY HOTTIE WHITE HEART,WH002,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 84029E, RED WOOLLY HOTTIE WHITE HEART, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 48 units. Current stock is 60 units. Average daily sales is 38.24 units. Days of inventory remaining is 1.26. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 195.0."
84029G_WH002,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,WH002,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 84029G, KNITTED UNION FLAG HOT WATER BOTTLE, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 25 units. Current stock is 35 units. Average daily sales is 17.91 units. Days of inventory remaining is 1.4. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 122.5."
22633_WH003,22633,HAND WARMER UNION JACK,WH003,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 22633, HAND WARMER UNION JACK, belongs to category Accessories and brand WarmHome. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 60 units. Current stock is 80 units. Average daily sales is 45.52 units. Days of inventory remaining is 1.32. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 300 units. Inventory value is 88.0."


In [0]:
display(
    retrieve_inventory_context("lead time", top_k=5)
)

document_id,stock_code,product_name,warehouse_id,stockout_risk_level,reorder_flag,business_priority_score,retrieval_score,final_score,rag_document_text
22752_WH003,22752,SET 7 BABUSHKA NESTING BOXES,WH003,High Risk,Reorder Needed,0.914,1.0,0.9656,"Product 22752, SET 7 BABUSHKA NESTING BOXES, belongs to category Gifts and brand GiftCraft. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 20 units. Current stock is 25 units. Average daily sales is 9.2 units. Days of inventory remaining is 2.17. Supplier is GiftCraft Wholesale with lead time of 12 days and reliability score of 0.86. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 100 units. Inventory value is 100.0."
84406B_WH002,84406B,CREAM CUPID HEARTS COAT HANGER,WH002,High Risk,Reorder Needed,0.912,1.0,0.9648,"Product 84406B, CREAM CUPID HEARTS COAT HANGER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 40 units. Current stock is 45 units. Average daily sales is 12.52 units. Days of inventory remaining is 3.19. Supplier is DecorCraft Europe with lead time of 10 days and reliability score of 0.88. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 150 units. Inventory value is 83.25."
84029E_WH002,84029E,RED WOOLLY HOTTIE WHITE HEART,WH002,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 84029E, RED WOOLLY HOTTIE WHITE HEART, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 48 units. Current stock is 60 units. Average daily sales is 38.24 units. Days of inventory remaining is 1.26. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 195.0."
84029G_WH002,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,WH002,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 84029G, KNITTED UNION FLAG HOT WATER BOTTLE, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 25 units. Current stock is 35 units. Average daily sales is 17.91 units. Days of inventory remaining is 1.4. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 122.5."
22633_WH003,22633,HAND WARMER UNION JACK,WH003,High Risk,Reorder Needed,0.909,1.0,0.9636,"Product 22633, HAND WARMER UNION JACK, belongs to category Accessories and brand WarmHome. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 60 units. Current stock is 80 units. Average daily sales is 45.52 units. Days of inventory remaining is 1.32. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 300 units. Inventory value is 88.0."


In [0]:
display(
    retrieve_inventory_context("85123A", top_k=5)
)

document_id,stock_code,product_name,warehouse_id,stockout_risk_level,reorder_flag,business_priority_score,retrieval_score,final_score,rag_document_text
85123A_WH001,85123A,WHITE HANGING HEART T-LIGHT HOLDER,WH001,High Risk,No Reorder Needed,0.606,1.0,0.8424,"Product 85123A, WHITE HANGING HEART T-LIGHT HOLDER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 210 units. Current stock is 250 units. Average daily sales is 120.15 units. Days of inventory remaining is 1.75. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is High Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 312.5."
22752_WH003,22752,SET 7 BABUSHKA NESTING BOXES,WH003,High Risk,Reorder Needed,0.914,0.1,0.0966,"Product 22752, SET 7 BABUSHKA NESTING BOXES, belongs to category Gifts and brand GiftCraft. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 20 units. Current stock is 25 units. Average daily sales is 9.2 units. Days of inventory remaining is 2.17. Supplier is GiftCraft Wholesale with lead time of 12 days and reliability score of 0.86. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 100 units. Inventory value is 100.0."
84406B_WH002,84406B,CREAM CUPID HEARTS COAT HANGER,WH002,High Risk,Reorder Needed,0.912,0.1,0.0965,"Product 84406B, CREAM CUPID HEARTS COAT HANGER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 40 units. Current stock is 45 units. Average daily sales is 12.52 units. Days of inventory remaining is 3.19. Supplier is DecorCraft Europe with lead time of 10 days and reliability score of 0.88. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 150 units. Inventory value is 83.25."
84029G_WH002,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,WH002,High Risk,Reorder Needed,0.909,0.1,0.0964,"Product 84029G, KNITTED UNION FLAG HOT WATER BOTTLE, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 25 units. Current stock is 35 units. Average daily sales is 17.91 units. Days of inventory remaining is 1.4. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 122.5."
84029E_WH002,84029E,RED WOOLLY HOTTIE WHITE HEART,WH002,High Risk,Reorder Needed,0.909,0.1,0.0964,"Product 84029E, RED WOOLLY HOTTIE WHITE HEART, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 48 units. Current stock is 60 units. Average daily sales is 38.24 units. Days of inventory remaining is 1.26. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 195.0."


In [0]:
def generate_simple_inventory_answer(query: str, top_k: int = 3):
    results = retrieve_inventory_context(query, top_k=top_k).toPandas()

    if results.empty:
        return "The available context does not contain enough information to answer this question reliably."

    answer_lines = []
    answer_lines.append(f"Question: {query}")
    answer_lines.append("")
    answer_lines.append("Relevant inventory context:")

    for _, row in results.iterrows():
        answer_lines.append("")
        answer_lines.append(
            f"- Product {row['stock_code']} ({row['product_name']}) "
            f"in warehouse {row['warehouse_id']} has stockout risk '{row['stockout_risk_level']}' "
            f"and reorder status '{row['reorder_flag']}'. "
            f"Business priority score: {row['business_priority_score']}."
        )

    answer_lines.append("")
    answer_lines.append("Recommendation:")
    answer_lines.append(
        "Review the highest-priority records first, especially products marked as High Risk or Reorder Needed."
    )

    return "\n".join(answer_lines)

In [0]:
print(generate_simple_inventory_answer("Which products should be reordered?", top_k=3))

Question: Which products should be reordered?

Relevant inventory context:

- Product 22752 (SET 7 BABUSHKA NESTING BOXES) in warehouse WH003 has stockout risk 'High Risk' and reorder status 'Reorder Needed'. Business priority score: 0.914.

- Product 84406B (CREAM CUPID HEARTS COAT HANGER) in warehouse WH002 has stockout risk 'High Risk' and reorder status 'Reorder Needed'. Business priority score: 0.912.

- Product 84029G (KNITTED UNION FLAG HOT WATER BOTTLE) in warehouse WH002 has stockout risk 'High Risk' and reorder status 'Reorder Needed'. Business priority score: 0.909.

Recommendation:
Review the highest-priority records first, especially products marked as High Risk or Reorder Needed.


# Retrieval Layer Completed

This notebook created a simple retrieval layer for the inventory RAG assistant.

The retrieval function searches the `inventory_rag_documents` table and ranks records using:

- Text relevance
- Stockout risk
- Reorder status
- Supplier reliability
- Business priority score

The ranking formula is:

```text
final_score = retrieval_score * (0.6 + 0.4 * business_priority_score)